# Lab 07 · A framework for choosing and checking

**Prepared by Mustafa Assaf** · 5–10 minute example

## Run the same steps in a framework

**So far:** we wrote each workflow by hand. In Lab 01, a loop called the model and then checked its answer.

**New:** **LangGraph**, a framework that names the parts of a workflow. *Nodes* are steps, *edges* set their order, and the *state* is the data the nodes share.

Our graph has two nodes. `choose` asks the model for a decision, and `check` tests that decision against the Python rule from Lab 00. Keeping generation and validation in separate nodes makes a disagreement easy to see. The framework does not make the model's choice correct.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'langgraph']):
    %pip install -q "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Write the two nodes

A node is a function that receives the current state and returns only the keys it adds. `choose` adds `decision`. `check` adds `matches_policy`, using Lab 00's rule written on one line.

In [ ]:
policy = "Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith."
def choose(state):
    reply = ask(f"{policy} Case: {state['case']}. Return only the choice.", example="modular monolith")
    return {"decision": reply.strip().lower()}
def check(state):
    case = state["case"]
    expected = "microservices" if case["team_size"] >= 8 and case["independent_deploys"] else "modular monolith"
    return {"matches_policy": state["decision"] == expected}

## Step 2 · Connect the nodes and run the graph

`TypedDict(...)` declares the three state keys: the input `case`, then `decision` and `matches_policy`. LangGraph merges each node's returned keys into the state. The edges set a fixed order: START → choose → check → END.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
graph = StateGraph(TypedDict("State", {"case": dict, "decision": str, "matches_policy": bool}))
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_edge(START, "choose")
graph.add_edge("choose", "check")
graph.add_edge("check", END)
result = graph.compile().invoke({"case": {"team_size": 3, "independent_deploys": False}})
print(result)
assert result["matches_policy"], "The model's choice failed the policy check."

## What you should see

The final state: `{'case': {...}, 'decision': 'modular monolith', 'matches_policy': True}`. The input supplied `case`, `choose` added `decision`, and `check` added `matches_policy`.

## Try it

Invoke the graph with the case `{"team_size": 8, "independent_deploys": True}` but keep the scripted reply `modular monolith`. Predict what `check` reports: the `assert` should stop with an error. Then change the scripted reply to `microservices`, or run live, and confirm that `matches_policy` is `True` again.

## Takeaway

This graph is a fixed workflow, not an autonomous agent: the edges always run choose, then check. LangGraph can also branch and loop, as Lab 04's loop did by hand. Passing a policy check is one piece of evidence. A real design review also weighs alternatives and trade-offs.

## Read further

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangGraph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)
- [LangGraph Complete Guide](https://github.com/mkassaf/langgraph-complete-guide): continue here after this lab.